In [ ]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

file = pd.read_csv('../../clean_COVIDSenti.csv')

def process_text(text):
    if not isinstance(text, str):
        return []
    tokens = word_tokenize(text)
    return [token for token in tokens if token not in stop_words]

sentences = file['tweet'].apply(process_text)
labels = file['label'].values
mask = sentences.apply(len) > 0
sentences = sentences[mask].tolist()
labels = labels[mask]
print(f"Data: {len(sentences)} sentences, {len(labels)} labels")

In [ ]:
import os
import time
import multiprocessing
import numpy as np
from gensim.models import Word2Vec

model_path = "word2vec.model"
features_path = "word2vec-features.npy"
labels_path = "word2vec-labels.npy"

all_sentences = []
for tokens in sentences:
    all_sentences.append(tokens if tokens else ['<pad'])
labels = labels[:len(all_sentences)]

if os.path.exists(model_path):
    print(f"Loading existing model from {model_path}...")
    w2v_model = Word2Vec.load(model_path)
    print(f"Model loaded. Vocab size: {len(w2v_model.wv)}")
else:
    print(f"Training Word2Vec (workers={multiprocessing.cpu_count()})...")
    start = time.time()
    w2v_model = Word2Vec(
        sg=1,
        epochs=60,
        vector_size=300,
        window=5,
        min_count=5,
        workers=multiprocessing.cpu_count(),
    )
    w2v_model.build_vocab(all_sentences)
    print(f"Vocab built: {len(w2v_model.wv)} words")
    w2v_model.train(all_sentences, total_examples=len(all_sentences), epochs=w2v_model.epochs)
    w2v_model.save(model_path)
    print(f"Model saved to {model_path} ({time.time()-start:.1f}s)")

In [ ]:
if os.path.exists(features_path) and os.path.exists(labels_path):
    print(f"Loading cached features from {features_path}...")
    X = np.load(features_path)
    y = np.load(labels_path)
    print(f"Features shape: {X.shape}, Labels shape: {y.shape}")
else:
    print(f"Extracting features for {len(all_sentences)} sentences...")
    start = time.time()
    vector_size = w2v_model.vector_size
    zero_vec = np.zeros(vector_size)
    X = np.zeros((len(all_sentences), vector_size), dtype=np.float32)
    for i, tokens in enumerate(all_sentences):
        vectors = [w2v_model.wv[w] for w in tokens if w in w2v_model.wv]
        X[i] = np.mean(vectors, axis=0) if vectors else zero_vec
        if (i + 1) % 10000 == 0:
            print(f"  Progress: {i+1}/{len(all_sentences)} ({time.time()-start:.1f}s)")
    y = np.array(labels)
    np.save(features_path, X)
    np.save(labels_path, y)
    print(f"Features saved. Shape: {X.shape} ({time.time()-start:.1f}s)")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

trainx, testx, trainy, testy = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {len(trainx)}, Test: {len(testx)}")
print(f"Label distribution: {dict(zip(*np.unique(y, return_counts=True)))}")

In [ ]:
import json

results_path = "rf_results.json"
results = []
if os.path.exists(results_path):
    with open(results_path, 'r') as f:
        results = json.load(f)
    print(f"Loaded {len(results)} cached results")

param_grid = [
    (est, dep, feat)
    for est in [200, 300, 500]
    for dep in [15, 20, None]
    for feat in ['sqrt', 'log2']
]

done_keys = {(r['n_estimators'], r['max_depth'], str(r['max_features'])) for r in results}
remaining = [(e, d, f) for e, d, f in param_grid if (e, d, str(f)) not in done_keys]
print(f"Total: {len(param_grid)}, Done: {len(results)}, Remaining: {len(remaining)}")

for idx, (est, dep, feat) in enumerate(remaining):
    print(f"\n[{len(results)+1}/{len(param_grid)}] n_estimators={est}, max_depth={dep}, max_features={feat}")
    start = time.time()
    rt = RandomForestClassifier(
        n_estimators=est,
        max_depth=dep,
        max_features=feat,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced',
    )
    rt.fit(trainx, trainy)
    predy = rt.predict(testx)
    acc = accuracy_score(testy, predy)
    elapsed = time.time() - start
    print(f"  Accuracy: {acc:.4f} ({elapsed:.1f}s)")
    print(classification_report(testy, predy, target_names=['Negative', 'Neutral', 'Positive']))

    results.append({
        'n_estimators': est,
        'max_depth': dep,
        'max_features': str(feat),
        'accuracy': acc,
        'elapsed': round(elapsed, 1),
    })
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)

print(f"\nAll done. {len(results)} results saved to {results_path}")

In [ ]:
best = max(results, key=lambda r: r['accuracy'])
print(f"Best: accuracy={best['accuracy']:.4f}")
print(f"  n_estimators={best['n_estimators']}, max_depth={best['max_depth']}, max_features={best['max_features']}")
print(f"\nAll results sorted by accuracy:")
for r in sorted(results, key=lambda r: r['accuracy'], reverse=True):
    print(f"  acc={r['accuracy']:.4f} | est={r['n_estimators']}, depth={r['max_depth']}, feat={r['max_features']}, time={r['elapsed']}s")